In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install tensorflow
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 78.4 MB/s eta 0:00:00


In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm

# KONFIGURASI
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 128
BATCH_SIZE = 64
ARTIFACTS_DIR = "/content/drive/My Drive/colab/bert_lg_artifacts"

# 1.LOAD & PREPARE DATA 
def prepare_data(file_path):
    print(f"[INFO] Loading and preparing data from {file_path}...")
    df = pd.read_csv(file_path)
    df = df.dropna(subset=['text', 'label', 'bahasa'])

    # Tambah prefix bahasa
    df['text'] = df['bahasa'].apply(lambda x: '[ID]' if x.lower() == 'in' else '[EN]') + ' ' + df['text']

    texts = df['text'].tolist()
    labels = df['label'].astype(int).tolist()

    X_train, X_val, y_train, y_val = train_test_split(
        texts, labels,
        test_size=0.2,
        stratify=labels,
        random_state=42
    )
    print(f"Train size: {len(X_train)}, Validation size: {len(X_val)}")
    return X_train, X_val, np.array(y_train), np.array(y_val)

# 2.HASILKAN EMBEDDING 
def get_bert_embeddings(texts, model, tokenizer):
    print(f"Generating BERT embeddings for {len(texts)} texts...")
    all_embeddings = []

    # Proses dalam batch untuk efisiensi memori
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch_texts = texts[i:i + BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="tf"
        )

        outputs = model(inputs)

        # ambil [CLS] token embedding (last_hidden_state[:, 0, :])
        cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)

# 3.SAVE ARTIFACTS
def save_artifacts(train_embeddings, val_embeddings, train_labels, model):
    print(f"Saving artifacts to {ARTIFACTS_DIR}...")
    os.makedirs(ARTIFACTS_DIR, exist_ok=True)

    #simpan embedding
    np.save(os.path.join(ARTIFACTS_DIR, "train_embeddings.npy"), train_embeddings)
    np.save(os.path.join(ARTIFACTS_DIR, "validation_embeddings.npy"), val_embeddings)
    print("Embedding saved sucess.")

    #simpan label
    joblib.dump(train_labels, os.path.join(ARTIFACTS_DIR, "train_labels.joblib"))
    print("Train labels saved sucess.")

    #simpan indeks FAISS
    joblib.dump(model, os.path.join(ARTIFACTS_DIR, "logreg_model.joblib"))
    print("Logistic Regression model saved sucess.")

# 4.LOGISTIC REGRESSION PROCESS
def train_logistic_regression(train_embeddings, y_train):
    print("Training Logistic Regression classifier...")
    clf = LogisticRegression(max_iter=2000, n_jobs=-1)
    clf.fit(train_embeddings, y_train)
    return clf

# RUN
def main(data_path):
    X_train, X_val, y_train, y_val = prepare_data(data_path)

    # Muat model BERT dasar dan tokenizer
    print(f"[INFO] Loading BERT model ({MODEL_NAME})...")
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    bert_model = TFBertModel.from_pretrained(MODEL_NAME, from_pt=True)

    # Hasilkan Embedding dan tipe data float32
    train_embeddings = get_bert_embeddings(X_train, bert_model, tokenizer).astype('float32')
    val_embeddings = get_bert_embeddings(X_val, bert_model, tokenizer).astype('float32')

    print(f"[INFO] Shape of train embeddings: {train_embeddings.shape}")
    print(f"[INFO] Shape of validation embeddings: {val_embeddings.shape}")

    # Training
    clf = train_logistic_regression(train_embeddings, y_train)

    # Simpan semua artefak
    save_artifacts(train_embeddings, val_embeddings, y_train, clf)

    # Evaluasi
    print("mulai evaluasi")
    start_time = time.time()
    predictions = clf.predict(val_embeddings)
    end_time = time.time()
    inference_time = end_time - start_time

    print("\n" + "="*20 + " HASIL EVALUASI " + "="*20)
    print(classification_report(y_val, predictions, target_names=['Label 0', 'Label 1']))
    print(f"Total Inference Time: {inference_time:.4f} seconds")
    avg_inference_time = (inference_time / len(y_val)) * 1000
    print(f"Rata-rata inference waktu per sampel: {avg_inference_time:.4f} ms")
    print("=" * 63)

# EKSEKUSI
if __name__ == "__main__":
    DATASET_PATH = "/content/drive/My Drive/dataset/4datasentimen_all_clean_ing_nolowercase_googlecolaboutlier34.csv"
    main(DATASET_PATH)

[INFO] Loading and preparing data from /content/drive/My Drive/dataset/4datasentimen_all_clean_ing_nolowercase_googlecolaboutlier34.csv...
[INFO] Data prepared. Train size: 5524, Validation size: 1381
[INFO] Loading BERT model (bert-base-multilingual-cased)...


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already

[INFO] Generating BERT embeddings for 5524 texts...


100%|██████████| 87/87 [00:19<00:00,  4.55it/s]


[INFO] Generating BERT embeddings for 1381 texts...


100%|██████████| 22/22 [00:04<00:00,  4.61it/s]


[INFO] Shape of train embeddings: (5524, 768)
[INFO] Shape of validation embeddings: (1381, 768)
[INFO] Training Logistic Regression classifier...
[INFO] Logistic Regression training complete.
[INFO] Saving artifacts to /content/drive/My Drive/colab/bert_lg_artifacts...
[INFO] Embeddings saved.
[INFO] Train labels saved.
[INFO] Logistic Regression model saved.
--------------------------------------------------
[INFO] Starting evaluation...

==================== EVALUATION REPORT ====================
              precision    recall  f1-score   support

     Label 0       0.89      0.92      0.90       860
     Label 1       0.86      0.81      0.83       521

    accuracy                           0.88      1381
   macro avg       0.87      0.86      0.87      1381
weighted avg       0.88      0.88      0.88      1381

Total Inference Time: 0.0021 seconds
Average Inference Time per Sample: 0.0015 ms


In [7]:
import os
import time
import joblib
import numpy as np
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer

# ======================== KONFIGURASI ========================
# PASTIKAN PATH INI SESUAI DENGAN LOKASI ARTEFAK ANDA DI GOOGLE DRIVE
ARTIFACTS_DIR = "/content/drive/My Drive/colab/bert_lg_artifacts"
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 128

# ======================== FUNGSI PREDIKSI ========================
def predict_single_text(text_to_predict, tokenizer, bert_model, logreg_model):
    """
    Fungsi untuk mengklasifikasikan satu buah teks menggunakan Logistic Regression.
    """
    print(f"Analyzing text: '{text_to_predict}'...")
    start_time = time.time()

    # 1. Buat embedding dari teks input
    inputs = tokenizer(
        [text_to_predict],
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="tf"
    )
    outputs = bert_model(inputs)
    embedding = outputs.last_hidden_state[:, 0, :].numpy().astype("float32")

    # 2. Prediksi dengan Logistic Regression
    predicted_label = logreg_model.predict(embedding)[0]
    probas = logreg_model.predict_proba(embedding)[0]
    confidence = np.max(probas)

    end_time = time.time()

    # 3. Tampilkan hasil analisis
    print("\n" + "="*20 + " ANALYSIS RESULT " + "="*20)
    print(f"Predicted Class: Label = {predicted_label}")
    print(f"Confidence: {confidence:.2%}")
    print(f"Probabilities: {probas}")
    print(f"Inference Time: {end_time - start_time:.4f} seconds")
    print("=" * 57)

    return predicted_label

# ======================== SCRIPT UTAMA ========================
if __name__ == "__main__":
    print("[INFO] Loading necessary artifacts, please wait...")

    try:
        # Muat tokenizer & BERT
        tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
        bert_model = TFBertModel.from_pretrained(MODEL_NAME, from_pt=True)

        # Muat model Logistic Regression
        logreg_path = os.path.join(ARTIFACTS_DIR, "logreg_model.joblib")
        logreg_model = joblib.load(logreg_path)

        print("[SUCCESS] All artifacts loaded. Ready to predict.")
        print("-" * 57)

        # Teks yang ingin dideteksi
        input_text = "stupid system"

        # Panggil fungsi prediksi
        predict_single_text(input_text, tokenizer, bert_model, logreg_model)

    except FileNotFoundError as e:
        print(f"\n[ERROR] Artifact file not found: {e}")
        print("Please make sure the path in ARTIFACTS_DIR is correct and your Google Drive is mounted.")
    except Exception as e:
        print(f"\n[ERROR] An unexpected error occurred: {e}")

[INFO] Loading necessary artifacts, please wait...


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already

[SUCCESS] All artifacts loaded. Ready to predict.
---------------------------------------------------------
Analyzing text: 'stupid system'...

==================== ANALYSIS RESULT ====================
Predicted Class: Label 1
Confidence: 90.55%
Probabilities: [0.09447574 0.90552426]
Inference Time: 0.1960 seconds
